# Brincar com os parâmetros do modelo

Painel interativo. Mude os controles, aperte **Rodar simulação**, veja o efeito. Cada rodada é uma execução completa do `WaaSModel` — não é animação pré-renderizada.

**Como usar este caderno:** clique no botão **Open in Colab** no [site](https://freirelucas.github.io/waas-antitrust/) ou execute em ambiente local com `pip install -e ".[dev]"`. A primeira célula instala o pacote no Colab; em ambiente local é um no-op.

**O que está exposto** (defaults vêm da calibração formal R03):

- **Regime** (A status quo · B Resolução CADE · C lei nova · EUA DOJ-ATR · UE DMA Tool) — qual configuração institucional.
- **Número de empresas** e **trabalhadores por empresa** — tamanho do sistema.
- **Fração de violadoras** — quantas empresas, no início, têm conduta anticompetitiva ativa.
- **Taxa de observação** — probabilidade de cada trabalhador ver a conduta no tique.
- **W_mult** — recompensa em múltiplos de salário anual.
- **k_rel** — fração de cooperadores necessária para massa crítica.
- **Erosão Coleman (alpha_erosao)** — risco de o substrato cooperativo erodir com instrumentalização (R26, forma fraca verificada empiricamente).
- **Canal explícito (R27)** — se o `AutoridadeAgent` mantém o escrow individual e abre simultaneamente quando atinge massa crítica.
- **Janela do escrow** — quantos tiques uma denúncia permanece em escrow antes de expirar (0 = eterno, leitura Callisto).

**O que sai:** painel 2×2 com (i) dano acumulado, (ii) número de violadoras ativas, (iii) sinais por tique, (iv) capital social residual. Linhas em verde são o regime escolhido; cinza é Regime A como referência.

In [ ]:
# Instala o pacote apenas no Google Colab (no-op em ambiente local/CI)
import sys

if "google.colab" in sys.modules:
    !pip install --quiet "waas-antitrust @ git+https://github.com/freirelucas/waas-antitrust.git@main"

In [ ]:
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

from waas_antitrust.model import WaaSModel, WaaSParametros
from waas_antitrust.viz import PALETA, aplicar_estilo

aplicar_estilo()


def rodar_e_plotar(
    regime: str,
    n_empresas: int,
    tam_medio_empresa: int,
    n_tiques: int,
    fracao_violadoras: float,
    taxa_observacao: float,
    W_mult: float,
    k_rel: float,
    alpha_erosao: float,
    usar_escrow_explicito: bool,
    janela_escrow_tiques: int,
    seed: int,
):
    # Roda o regime escolhido E o Regime A (referência) na mesma seed.
    config_comum = dict(
        n_empresas=n_empresas,
        tam_medio_empresa=tam_medio_empresa,
        n_tiques=n_tiques,
        seed=seed,
        fracao_violadoras=fracao_violadoras,
        taxa_observacao=taxa_observacao,
        W_mult=W_mult,
        k_rel=k_rel,
        alpha_erosao=alpha_erosao,
        usar_escrow_explicito=usar_escrow_explicito,
        janela_escrow_tiques=janela_escrow_tiques,
    )
    df_a = WaaSModel(WaaSParametros(regime="A", **config_comum)).executar()
    df_x = WaaSModel(WaaSParametros(regime=regime, **config_comum)).executar()

    fig, axes = plt.subplots(2, 2, figsize=(11, 7))
    cor_x = PALETA.get(regime, PALETA["B"])
    cor_a = PALETA["A"]
    eixos = [
        (axes[0, 0], "dano_acumulado", "(A) Dano acumulado"),
        (axes[0, 1], "n_violadoras_ativas", "(B) Violadoras ativas"),
        (axes[1, 0], "n_sinais", "(C) Sinais por tique"),
        (axes[1, 1], "capital_social_residual", "(D) Capital social residual"),
    ]
    for ax, coluna, titulo in eixos:
        ax.plot(df_a["tique"], df_a[coluna], color=cor_a, label="Regime A (referência)")
        ax.plot(df_x["tique"], df_x[coluna], color=cor_x, label=f"Regime {regime}")
        ax.set_title(titulo)
        ax.set_xlabel("Tique (trimestre)")
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)

    fig.suptitle(
        f"Regime {regime} vs A — {n_empresas} firmas × {n_tiques} tiques (seed {seed})",
        fontsize=11,
    )
    fig.tight_layout()
    plt.show()

    # Resumo numérico (último tique)
    print("\n=== Resumo (último tique) ===")
    print(f"  Dano A: {df_a['dano_acumulado'].iloc[-1]:8.1f}   " f"Dano {regime}: {df_x['dano_acumulado'].iloc[-1]:8.1f}")
    print(f"  Sinais A: {int(df_a['n_sinais'].sum()):6d}     " f"Sinais {regime}: {int(df_x['n_sinais'].sum()):6d}")
    print(f"  TCCs A: {int(df_a['n_tcc_assinados'].iloc[-1]):4d}        " f"TCCs {regime}: {int(df_x['n_tcc_assinados'].iloc[-1]):4d}")
    if usar_escrow_explicito:
        print(
            f"  Em escrow {regime}: {int(df_x['n_denuncias_em_escrow'].iloc[-1]):4d}   "
            f"Aberturas simultâneas {regime}: {int(df_x['n_aberturas_simultaneas_acum'].iloc[-1]):4d}"
        )

In [ ]:
regime_w = widgets.Dropdown(
    options=[("A — status quo (sem WaaS)", "A"), ("B — Resolução CADE", "B"), ("C — lei nova", "C"), ("EUA — DOJ-ATR Rewards", "EUA"), ("UE — DMA Tool", "UE")],
    value="B",
    description="Regime:",
    style={"description_width": "160px"},
)
n_empresas_w = widgets.IntSlider(value=10, min=4, max=40, step=2, description="N empresas:", style={"description_width": "160px"})
tam_w = widgets.IntSlider(value=120, min=30, max=400, step=10, description="Trab/empresa:", style={"description_width": "160px"})
n_tiques_w = widgets.IntSlider(value=20, min=8, max=60, step=2, description="Horizonte (tiques):", style={"description_width": "160px"})
fv_w = widgets.FloatSlider(value=0.5, min=0.1, max=0.9, step=0.05, description="Fração violadoras:", style={"description_width": "160px"})
obs_w = widgets.FloatSlider(value=0.45, min=0.05, max=0.8, step=0.05, description="Taxa observação:", style={"description_width": "160px"})
W_w = widgets.FloatSlider(value=1.5, min=0.0, max=4.0, step=0.25, description="W_mult (× w_a):", style={"description_width": "160px"})
k_w = widgets.FloatSlider(value=0.05, min=0.01, max=0.25, step=0.01, description="k_rel:", style={"description_width": "160px"})
alpha_w = widgets.FloatSlider(value=0.0, min=0.0, max=0.9, step=0.1, description="alpha_erosão:", style={"description_width": "160px"})
escrow_w = widgets.Checkbox(value=False, description="usar_escrow_explicito", style={"description_width": "160px"})
janela_w = widgets.IntSlider(value=0, min=0, max=12, step=1, description="janela_escrow_tiques:", style={"description_width": "160px"})
seed_w = widgets.IntSlider(value=11, min=1, max=999, step=1, description="Seed:", style={"description_width": "160px"})

botao = widgets.Button(description="Rodar simulação ▶", button_style="primary")
saida = widgets.Output()

def ao_clicar(_):
    with saida:
        clear_output(wait=True)
        rodar_e_plotar(
            regime=regime_w.value,
            n_empresas=n_empresas_w.value,
            tam_medio_empresa=tam_w.value,
            n_tiques=n_tiques_w.value,
            fracao_violadoras=fv_w.value,
            taxa_observacao=obs_w.value,
            W_mult=W_w.value,
            k_rel=k_w.value,
            alpha_erosao=alpha_w.value,
            usar_escrow_explicito=escrow_w.value,
            janela_escrow_tiques=janela_w.value,
            seed=seed_w.value,
        )

botao.on_click(ao_clicar)

controles = widgets.VBox([regime_w, n_empresas_w, tam_w, n_tiques_w, fv_w, obs_w, W_w, k_w, alpha_w, escrow_w, janela_w, seed_w, botao])
display(controles, saida)

# Roda uma vez automaticamente com defaults para não abrir vazio.
ao_clicar(None)

## Experimentos sugeridos

Cada um abaixo demora **menos de 30 segundos** depois do primeiro setup:

1. **Comparar Regime A vs B** — deixe o slider padrão, troque o regime para A e depois B. O dano em A cresce linear; em B achata.
2. **Forçar a Proposição 5 forte** — coloque alpha_erosão em 0.9 e Regime B. O dano deveria ainda ficar abaixo de A. Se romper, o achado da rodada de jun/2026 foi anulado — reabra `docs/limitacoes.md`.
3. **Ver o canal funcionando isoladamente** — Regime B + W_mult = 0 + usar_escrow_explicito = ON + alpha = 0. Os depósitos vêm dos arquétipos éticos e cascateiam por imitação.
4. **Comparar BR vs EUA vs UE** — alterne o regime entre B, EUA, UE com a mesma seed. UE replica A (sem recompensa, só proteção); EUA replica C (recompensa estatutária).
5. **Achar massa crítica inalcançável** — Regime B + k_rel = 0.25 + N empresas = 4. Nenhuma firma atinge o gatilho. Reduza k_rel para 0.05 e veja o sistema ganhar vida.

## Limites do brincar

Este painel é didático. Os achados científicos do projeto vêm de varreduras multi-seed (10+ sementes, bootstrap CI 95%), não de uma execução única. Para reproduzir os achados oficiais:

- Falsificação da Prop. 5 forte: `python scripts/varredura_alpha_erosao.py`
- Calibração formal R03: `python scripts/calibrar_formal.py`
- Identificabilidade: `python scripts/identificabilidade_r03.py`
- Todas as 19 figuras do site: `python scripts/regerar_todas_as_figuras.py`